# Assignment 1 — training pipeline

Preprocess Ames sales, then train the same linear model in small steps you can read one cell at a time.

`uv run python main.py train` runs every cell below with no manual steps.
Business case: `REPORT.md`. Preprocessing decisions: README.


## 1. Preprocess and split

Load Ames, fix date typos, drop the three partial outliers, fill gaps from 2006–2008 only, engineer ages, encode, scale. Target is `log(SalePrice)`.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("AmesHousing.csv")

# Impossible values, fixed with the same rule on every row.
df.loc[df["Garage Yr Blt"] == 2207, "Garage Yr Blt"] = 2007
remod_before_built = df["Year Remod/Add"] < df["Year Built"]
df.loc[remod_before_built, "Year Remod/Add"] = df.loc[remod_before_built, "Year Built"]
# A remodel year after the sale is not known yet.
future_remod = df["Year Remod/Add"] > df["Yr Sold"]
df.loc[future_remod, "Year Remod/Add"] = df.loc[future_remod, "Yr Sold"]

# Three partial sales of houses above 4,000 sq ft. De Cock: these prices are not market prices.
# The two other houses above 4,000 sq ft are kept: their prices match each other.
partial_outlier = (df["Gr Liv Area"] > 4000) & (df["Sale Condition"] == "Partial")
df = df.loc[~partial_outlier].copy()

# A blank means the house does not have that part.
no_basement = df["Bsmt Qual"].isna()
for col in ["Bsmt Qual", "Bsmt Cond", "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2"]:
    df.loc[no_basement, col] = "None"
for col in ["BsmtFin SF 1", "BsmtFin SF 2", "Bsmt Unf SF", "Total Bsmt SF", "Bsmt Full Bath", "Bsmt Half Bath"]:
    df.loc[no_basement, col] = 0

no_garage = df["Garage Type"].isna()
for col in ["Garage Type", "Garage Finish", "Garage Qual", "Garage Cond"]:
    df.loc[no_garage, col] = "None"
df.loc[no_garage, "Garage Area"] = 0

df["Mas Vnr Type"] = df["Mas Vnr Type"].fillna("None")
no_veneer = df["Mas Vnr Type"].eq("None") & df["Mas Vnr Area"].isna()
df.loc[no_veneer, "Mas Vnr Area"] = 0

for col in ["Alley", "Fireplace Qu", "Fence"]:
    df[col] = df[col].fillna("None")

df["MS SubClass"] = df["MS SubClass"].astype(str)
df["Mo Sold"] = df["Mo Sold"].astype(str).str.zfill(2)

# Price a sale using only earlier sales. 2010 stops in July.
train = df[df["Yr Sold"] <= 2008].copy()
val = df[df["Yr Sold"] == 2009].copy()
test = df[df["Yr Sold"] == 2010].copy()

# Fill true gaps from the training years only.
frontage_by_neighborhood = train.groupby("Neighborhood")["Lot Frontage"].median()
frontage_median = train["Lot Frontage"].median()
veneer_area_by_type = train.groupby("Mas Vnr Type")["Mas Vnr Area"].median()
veneer_area_median = train["Mas Vnr Area"].median()

detached = train["Garage Type"].eq("Detchd")
detached_fill = {
    "Garage Finish": train.loc[detached, "Garage Finish"].mode().iloc[0],
    "Garage Qual": train.loc[detached, "Garage Qual"].mode().iloc[0],
    "Garage Cond": train.loc[detached, "Garage Cond"].mode().iloc[0],
    "Garage Yr Blt": train.loc[detached, "Garage Yr Blt"].median(),
    "Garage Area": train.loc[detached, "Garage Area"].median(),
}

with_basement = train["Bsmt Qual"].ne("None")
basement_fill = {
    "Bsmt Exposure": train.loc[with_basement, "Bsmt Exposure"].mode().iloc[0],
    "BsmtFin Type 2": train.loc[with_basement, "BsmtFin Type 2"].mode().iloc[0],
}
electrical_mode = train["Electrical"].mode().iloc[0]


def fill_gaps(frame):
    frame = frame.copy()
    frame["Lot Frontage"] = frame["Lot Frontage"].fillna(
        frame["Neighborhood"].map(frontage_by_neighborhood)
    )
    frame["Lot Frontage"] = frame["Lot Frontage"].fillna(frontage_median)
    frame["Mas Vnr Area"] = frame["Mas Vnr Area"].fillna(
        frame["Mas Vnr Type"].map(veneer_area_by_type)
    )
    frame["Mas Vnr Area"] = frame["Mas Vnr Area"].fillna(veneer_area_median)
    for col, value in detached_fill.items():
        frame[col] = frame[col].fillna(value)
    for col, value in basement_fill.items():
        frame[col] = frame[col].fillna(value)
    frame["Electrical"] = frame["Electrical"].fillna(electrical_mode)
    return frame


train = fill_gaps(train)
val = fill_gaps(val)
test = fill_gaps(test)


def add_ages(frame):
    frame = frame.copy()
    frame["Age"] = frame["Yr Sold"] - frame["Year Built"]
    frame["RemodAge"] = frame["Yr Sold"] - frame["Year Remod/Add"]
    # 0 when there is no garage. A raw year would invent a construction date.
    frame["GarageAge"] = 0.0
    has_garage = frame["Garage Type"].ne("None")
    frame.loc[has_garage, "GarageAge"] = (
        frame.loc[has_garage, "Yr Sold"] - frame.loc[has_garage, "Garage Yr Blt"]
    )
    return frame


train = add_ages(train)
val = add_ages(val)
test = add_ages(test)

QUALITY = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
FINISH = {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}
ORDINAL = {
    "Exter Qual": QUALITY,
    "Exter Cond": QUALITY,
    "Bsmt Qual": QUALITY,
    "Bsmt Cond": QUALITY,
    "Heating QC": QUALITY,
    "Kitchen Qual": QUALITY,
    "Fireplace Qu": QUALITY,
    "Garage Qual": QUALITY,
    "Garage Cond": QUALITY,
    "Bsmt Exposure": {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFin Type 1": FINISH,
    "BsmtFin Type 2": FINISH,
    "Functional": {"Sal": 0, "Sev": 1, "Maj2": 2, "Maj1": 3, "Mod": 4, "Min2": 5, "Min1": 6, "Typ": 7},
    "Garage Finish": {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "Lot Shape": {"IR3": 0, "IR2": 1, "IR1": 2, "Reg": 3},
    "Land Slope": {"Gtl": 0, "Mod": 1, "Sev": 2},
    "Paved Drive": {"N": 0, "P": 1, "Y": 2},
    "Fence": {"None": 0, "MnWw": 1, "GdWo": 2, "MnPrv": 3, "GdPrv": 4},
    "Central Air": {"N": 0, "Y": 1},
}

NOMINAL = [
    "MS SubClass", "MS Zoning", "Alley", "Lot Config", "Land Contour",
    "Neighborhood", "Condition 1", "Bldg Type", "House Style", "Roof Style",
    "Exterior 1st", "Mas Vnr Type", "Foundation", "Garage Type", "Electrical",
    "Mo Sold",
]

DROP = [
    "Order", "PID", "Street", "Utilities", "Condition 2", "Roof Matl", "Heating",
    "Exterior 2nd", "Pool QC", "Misc Feature", "Sale Type", "Sale Condition",
    "1st Flr SF", "2nd Flr SF", "Low Qual Fin SF", "Bsmt Unf SF", "Garage Cars",
    "Year Built", "Year Remod/Add", "Garage Yr Blt", "SalePrice",
]


def encode_ordinal(frame):
    frame = frame.copy()
    for col, mapping in ORDINAL.items():
        frame[col] = frame[col].map(mapping)
    return frame


train = encode_ordinal(train)
val = encode_ordinal(val)
test = encode_ordinal(test)

# Fewer than 10 training rows, and the level is renamed Other in later years too.
for col in NOMINAL:
    common = set(train[col].value_counts().loc[lambda s: s >= 10].index)
    for frame in (train, val, test):
        frame[col] = frame[col].where(frame[col].isin(common), "Other")


def matrix(frame):
    features = frame.drop(columns=DROP)
    still_missing = features.columns[features.isna().any()].tolist()
    if still_missing:
        raise ValueError(f"Missing values left in {still_missing}")
    y = np.log(frame["SalePrice"].to_numpy(dtype=float))
    return features, y


X_train, y_train = matrix(train)
X_val, y_val = matrix(val)
X_test, y_test = matrix(test)

text_left = X_train.select_dtypes(exclude="number").columns.difference(NOMINAL)
if len(text_left) > 0:
    raise ValueError(f"Columns not encoded: {text_left.tolist()}")

# Alphabetical order, then leave out the first level. The intercept stands in for it.
for col in NOMINAL:
    categories = sorted(X_train[col].astype(str).unique())
    for frame in (X_train, X_val, X_test):
        frame[col] = pd.Categorical(frame[col], categories=categories)


def align(features):
    numeric = features.drop(columns=NOMINAL).reset_index(drop=True)
    dummies = pd.get_dummies(features[NOMINAL], drop_first=True, dtype=float)
    return pd.concat([numeric, dummies.reset_index(drop=True)], axis=1)


X_train = align(X_train)
X_val = align(X_val)
X_test = align(X_test)
if not (list(X_train.columns) == list(X_val.columns) == list(X_test.columns)):
    raise ValueError("Train, validation, and test columns differ")
feature_names = list(X_train.columns)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names)
X_val = pd.DataFrame(scaler.transform(X_val), columns=feature_names)
X_test = pd.DataFrame(scaler.transform(X_test), columns=feature_names)

if X_train.isna().any().any() or X_val.isna().any().any() or X_test.isna().any().any():
    raise ValueError("Missing values left after scaling")

print(f"Rows: train {len(y_train)}, validation {len(y_val)}, test {len(y_test)}")
print(f"Features: {X_train.shape[1]}")
print(f"Training geometric mean price: ${np.exp(y_train.mean()):,.0f}")


## 2. Training helpers

Arrays for the three splits, dollar metrics after undoing the log, and a small printer for coefficients.


In [ ]:
import json
from pathlib import Path

import joblib
import torch
from sklearn.linear_model import Lasso, LinearRegression, Ridge

ART = Path("artifacts")
ART.mkdir(exist_ok=True)

X_tr = X_train.to_numpy(dtype=np.float64)
X_va = X_val.to_numpy(dtype=np.float64)
X_te = X_test.to_numpy(dtype=np.float64)
y_tr, y_va, y_te = y_train, y_val, y_test
n_features = X_tr.shape[1]


def dollar_metrics(y_log_true, y_log_pred):
    actual = np.exp(y_log_true)
    pred = np.exp(y_log_pred)
    err = pred - actual
    return {
        "rmse": float(np.sqrt(np.mean(err**2))),
        "mae": float(np.mean(np.abs(err))),
    }


def eval_splits(predict_log):
    return {
        "train": dollar_metrics(y_tr, predict_log(X_tr)),
        "val": dollar_metrics(y_va, predict_log(X_va)),
        "test": dollar_metrics(y_te, predict_log(X_te)),
    }


def print_metrics(name, splits):
    print(f"{name}")
    print(f"{'split':<6} {'$RMSE':>12} {'$MAE':>12}")
    for split, m in splits.items():
        print(f"{split:<6} {m['rmse']:>12,.0f} {m['mae']:>12,.0f}")
    print()


def print_linear_details(name, coef, intercept, *, alpha=None, lr=None, epochs=None):
    print(f"{name}")
    print(f"  features: {len(coef)}")
    print(f"  intercept (log price): {intercept:.6f}  ->  ${np.exp(intercept):,.0f}")
    print(f"  coef: mean={coef.mean():.4e}, std={coef.std():.4e}, "
          f"min={coef.min():.4e}, max={coef.max():.4e}")
    if alpha is not None:
        print(f"  alpha (L2): {alpha}")
    if lr is not None:
        print(f"  learning rate: {lr}")
    if epochs is not None:
        print(f"  epochs: {epochs}")
    order = np.argsort(np.abs(coef))[::-1]
    print("  top |coef|:")
    for i in order[:8]:
        print(f"    {feature_names[i]:<28} {coef[i]:+.5f}")
    print()


print(f"Ready: train {len(y_tr)}, val {len(y_va)}, test {len(y_te)}, features {n_features}")


## 3. Naive baseline

Predict the training geometric mean for every house. This is the floor the linear models must beat.


In [ ]:
baseline_log = float(y_tr.mean())
baseline_splits = eval_splits(lambda X: np.full(len(X), baseline_log))

print(f"Baseline prediction: exp(mean log price) = ${np.exp(baseline_log):,.0f}")
print("(same number for every row)\n")
print_metrics("Naive baseline", baseline_splits)


## 4. OLS — scikit-learn

Closed-form ordinary least squares. No penalty. Reference solution the PyTorch loops should approach.


In [ ]:
sk_ols = LinearRegression().fit(X_tr, y_tr)
sk_ols_splits = eval_splits(lambda X: sk_ols.predict(X))

print_linear_details("sklearn LinearRegression (OLS)", sk_ols.coef_, float(sk_ols.intercept_))
print_metrics("sklearn OLS", sk_ols_splits)
print(
    f"Val $RMSE vs baseline: "
    f"{baseline_splits['val']['rmse']:,.0f} -> {sk_ols_splits['val']['rmse']:,.0f}"
)


## 5. OLS — manual PyTorch

No `nn.Module`, no `optim`. Weights `w`, bias `b`, prediction `X @ w + b`, mean squared error on log price, full-batch SGD by hand.


In [ ]:
def train_manual_pytorch(X, y, *, lr, epochs, alpha=0.0, seed=0, log_every=None):
    """Hand-written linear model: X @ w + b, MSE (+ optional L2), SGD."""
    torch.manual_seed(seed)
    Xt = torch.tensor(X, dtype=torch.float64)
    yt = torch.tensor(y, dtype=torch.float64)
    w = torch.zeros(X.shape[1], dtype=torch.float64)
    b = torch.zeros((), dtype=torch.float64)
    n = Xt.shape[0]
    if log_every is None:
        log_every = max(epochs // 5, 1)
    history = []
    for epoch in range(1, epochs + 1):
        pred = Xt @ w + b
        residual = pred - yt
        mse = (residual * residual).mean()
        # (alpha/n)||w||^2 matches sklearn Ridge SSE + alpha||w||^2 up to 1/n.
        loss = mse + (alpha / n) * (w * w).sum()
        grad_w = (Xt.T @ residual) * (2.0 / n) + (2.0 * alpha / n) * w
        grad_b = residual.mean() * 2.0
        w = w - lr * grad_w
        b = b - lr * grad_b
        if epoch == 1 or epoch % log_every == 0 or epoch == epochs:
            history.append((epoch, float(mse.detach()), float(loss.detach())))
    return w.detach().numpy(), float(b.detach()), history


OLS_LR = 0.05
OLS_EPOCHS = 8000
w_manual_ols, b_manual_ols, hist_manual_ols = train_manual_pytorch(
    X_tr, y_tr, lr=OLS_LR, epochs=OLS_EPOCHS, alpha=0.0
)
manual_ols_splits = eval_splits(lambda X: X @ w_manual_ols + b_manual_ols)

print_linear_details(
    "manual PyTorch OLS",
    w_manual_ols,
    b_manual_ols,
    alpha=0.0,
    lr=OLS_LR,
    epochs=OLS_EPOCHS,
)
print("Loss trail (epoch, MSE, loss):")
for epoch, mse, loss in hist_manual_ols:
    print(f"  {epoch:>5}  mse={mse:.6f}  loss={loss:.6f}")
print()
print_metrics("manual PyTorch OLS", manual_ols_splits)
coef_gap = np.max(np.abs(sk_ols.coef_ - w_manual_ols))
print(f"Max |coef| gap vs sklearn OLS: {coef_gap:.4e}")
print(f"|intercept| gap vs sklearn OLS: {abs(float(sk_ols.intercept_) - b_manual_ols):.4e}")


## 6. OLS — standard PyTorch

Same objective, but with `nn.Linear` and `torch.optim.SGD`. Still full-batch, same learning rate and epoch count as the manual loop.


In [ ]:
def train_standard_pytorch(X, y, *, lr, epochs, alpha=0.0, seed=0, log_every=None):
    torch.manual_seed(seed)
    Xt = torch.tensor(X, dtype=torch.float64)
    yt = torch.tensor(y, dtype=torch.float64).unsqueeze(1)
    n = Xt.shape[0]
    model = torch.nn.Linear(X.shape[1], 1, bias=True, dtype=torch.float64)
    torch.nn.init.zeros_(model.weight)
    torch.nn.init.zeros_(model.bias)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    if log_every is None:
        log_every = max(epochs // 5, 1)
    history = []
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        pred = model(Xt)
        mse = torch.nn.functional.mse_loss(pred, yt)
        loss = mse + (alpha / n) * model.weight.pow(2).sum()
        loss.backward()
        optimizer.step()
        if epoch == 1 or epoch % log_every == 0 or epoch == epochs:
            history.append((epoch, float(mse.detach()), float(loss.detach())))
    w = model.weight.detach().numpy().ravel()
    b = float(model.bias.detach())
    return model, w, b, history


model_std_ols, w_std_ols, b_std_ols, hist_std_ols = train_standard_pytorch(
    X_tr, y_tr, lr=OLS_LR, epochs=OLS_EPOCHS, alpha=0.0
)
std_ols_splits = eval_splits(lambda X: X @ w_std_ols + b_std_ols)

print_linear_details(
    "standard PyTorch OLS (nn.Linear + optim.SGD)",
    w_std_ols,
    b_std_ols,
    alpha=0.0,
    lr=OLS_LR,
    epochs=OLS_EPOCHS,
)
print("Loss trail (epoch, MSE, loss):")
for epoch, mse, loss in hist_std_ols:
    print(f"  {epoch:>5}  mse={mse:.6f}  loss={loss:.6f}")
print()
print_metrics("standard PyTorch OLS", std_ols_splits)
print(f"Max |coef| gap vs sklearn OLS: {np.max(np.abs(sk_ols.coef_ - w_std_ols)):.4e}")
print(f"Max |coef| gap vs manual PyTorch: {np.max(np.abs(w_manual_ols - w_std_ols)):.4e}")


## 7. OLS three-way check

All three unregularized fits on the same split, next to the baseline. They should nearly agree.


In [ ]:
print(f"{'model':<22} {'split':<6} {'$RMSE':>12} {'$MAE':>12}")
print("-" * 54)
for name, splits in [
    ("baseline", baseline_splits),
    ("sklearn OLS", sk_ols_splits),
    ("manual PyTorch OLS", manual_ols_splits),
    ("standard PyTorch OLS", std_ols_splits),
]:
    for split, m in splits.items():
        print(f"{name:<22} {split:<6} {m['rmse']:>12,.0f} {m['mae']:>12,.0f}")


## 8. Ridge and Lasso on validation

OLS can overfit 159 scaled columns. Sweep Ridge and Lasso on 2009 dollar RMSE.

The three graded implementations must be the **same** model. Lasso needs a proximal step the manual loop does not have, so the shared family is OLS or Ridge. Lasso is only a reference here.


In [ ]:
ridge_alphas = [0.01, 0.1, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0, 1000.0]
lasso_alphas = [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 0.1]

print("Ridge sweep (sklearn)")
print(f"{'alpha':>10} {'val $RMSE':>12} {'val $MAE':>12} {'nnz':>6}")
ridge_val = []
for a in ridge_alphas:
    model = Ridge(alpha=a, fit_intercept=True).fit(X_tr, y_tr)
    m = dollar_metrics(y_va, model.predict(X_va))
    nnz = int(np.sum(np.abs(model.coef_) > 1e-8))
    ridge_val.append((a, m["rmse"], m["mae"], model))
    print(f"{a:>10g} {m['rmse']:>12,.0f} {m['mae']:>12,.0f} {nnz:>6}")

print("\nLasso sweep (sklearn, reference only)")
print(f"{'alpha':>10} {'val $RMSE':>12} {'val $MAE':>12} {'nnz':>6}")
lasso_val = []
for a in lasso_alphas:
    model = Lasso(alpha=a, fit_intercept=True, max_iter=50000).fit(X_tr, y_tr)
    m = dollar_metrics(y_va, model.predict(X_va))
    nnz = int(np.sum(np.abs(model.coef_) > 1e-8))
    lasso_val.append((a, m["rmse"], m["mae"], model))
    print(f"{a:>10g} {m['rmse']:>12,.0f} {m['mae']:>12,.0f} {nnz:>6}")

best_ridge = min(ridge_val, key=lambda t: t[1])
best_lasso = min(lasso_val, key=lambda t: t[1])
ols_val_rmse = sk_ols_splits["val"]["rmse"]

print("\nSummary on 2009 validation")
print(f"  OLS:                    ${ols_val_rmse:,.0f} RMSE")
print(f"  best Ridge alpha={best_ridge[0]}: ${best_ridge[1]:,.0f} RMSE")
print(f"  best Lasso alpha={best_lasso[0]}: ${best_lasso[1]:,.0f} RMSE")

if best_ridge[1] < ols_val_rmse:
    chosen_family = "ridge"
    chosen_alpha = float(best_ridge[0])
    sk_final = best_ridge[3]
else:
    chosen_family = "ols"
    chosen_alpha = 0.0
    sk_final = sk_ols

torch_alpha = chosen_alpha
print(f"\nShared three-way model: {chosen_family} (alpha={chosen_alpha})")
if best_lasso[1] + 1e-9 < best_ridge[1]:
    print(
        f"Note: Lasso alone was better on val (${best_lasso[1]:,.0f}) "
        "but is not used for the three-way comparison."
    )


## 9. sklearn final model details

Refit is already done inside the sweep; this cell only inspects the chosen sklearn model.


In [ ]:
sk_final_splits = eval_splits(lambda X: sk_final.predict(X))
print_linear_details(
    f"sklearn {chosen_family}",
    sk_final.coef_,
    float(sk_final.intercept_),
    alpha=chosen_alpha,
)
print_metrics(f"sklearn {chosen_family}", sk_final_splits)
print(
    f"Val $RMSE: OLS {ols_val_rmse:,.0f} -> {chosen_family} {sk_final_splits['val']['rmse']:,.0f}"
)
print(
    f"Test $RMSE: OLS {sk_ols_splits['test']['rmse']:,.0f} -> "
    f"{chosen_family} {sk_final_splits['test']['rmse']:,.0f}"
)


## 10. Learning-rate sweep (PyTorch, same alpha)

Pick LR on 2009 for both PyTorch implementations. Drop runs that diverge to NaN.


In [ ]:
lr_grid = [0.001, 0.005, 0.01, 0.05, 0.1]
TUNE_EPOCHS = 4000

print(f"Penalty for PyTorch: alpha={torch_alpha}")
print(f"{'lr':>8} {'manual val $RMSE':>18} {'standard val $RMSE':>20}")
lr_results = []
for lr in lr_grid:
    w_m, b_m, _ = train_manual_pytorch(
        X_tr, y_tr, lr=lr, epochs=TUNE_EPOCHS, alpha=torch_alpha
    )
    _, w_s, b_s, _ = train_standard_pytorch(
        X_tr, y_tr, lr=lr, epochs=TUNE_EPOCHS, alpha=torch_alpha
    )
    rmse_m = dollar_metrics(y_va, X_va @ w_m + b_m)["rmse"]
    rmse_s = dollar_metrics(y_va, X_va @ w_s + b_s)["rmse"]
    lr_results.append((lr, rmse_m, rmse_s))
    m_txt = f"{rmse_m:,.0f}" if np.isfinite(rmse_m) else "NaN (diverged)"
    s_txt = f"{rmse_s:,.0f}" if np.isfinite(rmse_s) else "NaN (diverged)"
    print(f"{lr:>8g} {m_txt:>18} {s_txt:>20}")

lr_finite = [
    (lr, rm, rs) for lr, rm, rs in lr_results if np.isfinite(rm) and np.isfinite(rs)
]
if not lr_finite:
    raise RuntimeError("All learning rates diverged")

best_lr_manual = min(lr_finite, key=lambda t: t[1])[0]
best_lr_standard = min(lr_finite, key=lambda t: t[2])[0]
print(f"\nChosen LR - manual: {best_lr_manual}")
print(f"Chosen LR - standard: {best_lr_standard}")


## 11. Final manual PyTorch

Retrain with the chosen alpha and learning rate for more epochs.


In [ ]:
FINAL_EPOCHS = 10000
w_manual, b_manual, hist_manual = train_manual_pytorch(
    X_tr, y_tr, lr=best_lr_manual, epochs=FINAL_EPOCHS, alpha=torch_alpha
)
manual_splits = eval_splits(lambda X: X @ w_manual + b_manual)

print_linear_details(
    f"manual PyTorch {chosen_family}",
    w_manual,
    b_manual,
    alpha=torch_alpha,
    lr=best_lr_manual,
    epochs=FINAL_EPOCHS,
)
print("Loss trail (epoch, MSE, loss):")
for epoch, mse, loss in hist_manual:
    print(f"  {epoch:>5}  mse={mse:.6f}  loss={loss:.6f}")
print()
print_metrics(f"manual PyTorch {chosen_family}", manual_splits)
print(f"Max |coef| gap vs sklearn: {np.max(np.abs(sk_final.coef_ - w_manual)):.4e}")


## 12. Final standard PyTorch

Same alpha / comparable LR, `nn.Linear` + `optim.SGD`.


In [ ]:
model_standard, w_standard, b_standard, hist_standard = train_standard_pytorch(
    X_tr, y_tr, lr=best_lr_standard, epochs=FINAL_EPOCHS, alpha=torch_alpha
)
standard_splits = eval_splits(lambda X: X @ w_standard + b_standard)

print_linear_details(
    f"standard PyTorch {chosen_family}",
    w_standard,
    b_standard,
    alpha=torch_alpha,
    lr=best_lr_standard,
    epochs=FINAL_EPOCHS,
)
print("Loss trail (epoch, MSE, loss):")
for epoch, mse, loss in hist_standard:
    print(f"  {epoch:>5}  mse={mse:.6f}  loss={loss:.6f}")
print()
print_metrics(f"standard PyTorch {chosen_family}", standard_splits)
print(f"Max |coef| gap vs sklearn: {np.max(np.abs(sk_final.coef_ - w_standard)):.4e}")
print(f"Max |coef| gap vs manual: {np.max(np.abs(w_manual - w_standard)):.4e}")


## 13. Final comparison table

Baseline + three saved models. Test column is the one that matters for the report; val was only for choosing alpha and LR.


In [ ]:
final_rows = {
    "baseline": baseline_splits,
    f"sklearn {chosen_family}": sk_final_splits,
    "manual PyTorch": manual_splits,
    "standard PyTorch": standard_splits,
}

print(f"{'model':<22} {'split':<6} {'$RMSE':>12} {'$MAE':>12}")
print("-" * 54)
for name, splits in final_rows.items():
    for split, m in splits.items():
        print(f"{name:<22} {split:<6} {m['rmse']:>12,.0f} {m['mae']:>12,.0f}")


## 14. Save artifacts

Gradio loads these files and never retrains.


In [ ]:
joblib.dump(scaler, ART / "scaler.joblib")
joblib.dump(sk_final, ART / "sklearn_model.joblib")
np.save(ART / "X_train.npy", X_tr)
np.save(ART / "X_val.npy", X_va)
np.save(ART / "X_test.npy", X_te)
np.save(ART / "y_train.npy", y_tr)
np.save(ART / "y_val.npy", y_va)
np.save(ART / "y_test.npy", y_te)

torch.save(
    {
        "w": torch.tensor(w_manual),
        "b": b_manual,
        "alpha": torch_alpha,
        "lr": best_lr_manual,
        "epochs": FINAL_EPOCHS,
    },
    ART / "pytorch_manual.pt",
)
torch.save(
    {
        "state_dict": model_standard.state_dict(),
        "alpha": torch_alpha,
        "lr": best_lr_standard,
        "epochs": FINAL_EPOCHS,
        "n_features": n_features,
    },
    ART / "pytorch_standard.pt",
)


def _to_plain(obj):
    if isinstance(obj, dict):
        return {k: _to_plain(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_plain(v) for v in obj]
    if isinstance(obj, (np.floating, float)):
        return float(obj)
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    return obj


metrics = {
    "baseline_log_mean": baseline_log,
    "baseline_dollar": float(np.exp(baseline_log)),
    "chosen_family": chosen_family,
    "chosen_alpha": chosen_alpha,
    "torch_alpha": torch_alpha,
    "lr_manual": best_lr_manual,
    "lr_standard": best_lr_standard,
    "ols": {
        "baseline": baseline_splits,
        "sklearn": sk_ols_splits,
        "manual": manual_ols_splits,
        "standard": std_ols_splits,
    },
    "final": final_rows,
    "ridge_val_rmse": {str(a): r for a, r, _, _ in ridge_val},
    "lasso_val_rmse": {str(a): r for a, r, _, _ in lasso_val},
}

with open(ART / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(_to_plain(metrics), f, indent=2)
with open(ART / "feature_names.json", "w", encoding="utf-8") as f:
    json.dump(feature_names, f, indent=2)
with open(ART / "baseline.json", "w", encoding="utf-8") as f:
    json.dump(
        {"log_mean": baseline_log, "dollar": float(np.exp(baseline_log))},
        f,
        indent=2,
    )

print(f"Wrote artifacts to {ART.resolve()}")
print("Files:", ", ".join(sorted(p.name for p in ART.iterdir())))
